# Chapter 09: The Event Model (Reference)

## Learning Objectives

- Print the full event comparison matrix from the resources cheatsheet
- Explain what pull_request vs pull_request_target each expose
- Simulate the workflow_run two-hop head_sha collapse
- State why this repo's automerge.yml no longer chains past one hop

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [ ]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

## 1. The Event Comparison Matrix

The next cell prints `EVENT_MATRIX` as a table. You should see five rows: pull_request, pull_request_target, schedule, workflow_dispatch, workflow_run, each with its secret-availability and fork-PR-reach columns.

In [ ]:
from labs.lab_09_event_matrix import print_event_matrix, describe_event

print_event_matrix()

## 2. Look Up One Event

The next cell looks up `pull_request_target` specifically. You should see its checked-out ref described as the BASE branch's code, not the PR's.

In [ ]:
row = describe_event("pull_request_target")
for key, value in row.items():
    print(f"{key}: {value}")

## 3. The workflow_run Chaining Trap, Simulated

The next cell simulates `head_sha` fidelity across a 3-hop `workflow_run` chain. You should see hop 1 report the real PR SHA, and hops 2-3 report `COLLAPSED_TO_DEFAULT_BRANCH_SHA` -- exactly what this repo's own build hit.

In [ ]:
from labs.lab_09_event_matrix import simulate_workflow_run_chain

for hop, sha in enumerate(simulate_workflow_run_chain(3), start=1):
    print(f"hop {hop}: head_sha = {sha}")

## 4. Real Runs: Which Event Fired Each, and Its `head_sha`

The next cell lists the repo's recent workflow runs via `GET /repos/{owner}/{repo}/actions/runs` (`gh api`). In fixture mode it reads `fixtures/workflow_runs_sample.json` -- the PR #101 spine, where CI and Gate 3 fire on `pull_request` and Gate 2 fires on `workflow_run` off CI. Why it matters: this is the chaining trap in real data. You should see every run carrying the same `head_sha` (`a1b2c3d4e5f6…`), because Gate 2 is a **first** hop.

Re-run with `PRA_MODE=live PRA_REPO=<you>/practice_git_intro_to_pr_automerge` to see the sandbox repo's real runs instead; the same pattern holds.


In [ ]:
from labs.lab_09_event_matrix import list_recent_runs, print_recent_runs

repo = PRA_REPO or "octocat/practice_git_intro_to_pr_automerge"
runs = list_recent_runs(repo)
print_recent_runs(runs)
print(f"\n{len(runs)} runs; distinct head_sha values: {len({r['head_sha'] for r in runs})}")


## Takeaways & Next Steps

This notebook's takeaway is the hop-2 collapse above -- re-read Chapter 09 Section 8 against this printed output before moving on.

In [ ]:
print("See resources/gha_event_reference.md for the full, unabridged comparison.")

---

📖 **Reading companion:** [Chapter 09: The Event Model](../learning_modules/chapter_09_event_model.md)
🔬 **Try it live:** Re-run this notebook with `PRA_MODE=live PRA_REPO=<you>/practice_git_intro_to_pr_automerge`
